This script fetches live ADS-B data from [adsb.fi](https://adsb.fi/) for personal, non-commercial use. Getting historical data is harder than getting live data. But getting live data is slow.

In [ ]:
import subprocess
import time
import json
import sqlite3
import pandas as pd
# import csv
import os

In [ ]:
start_time = time.time()
command = ['curl', '--fail', 'https://opendata.adsb.fi/api/v3/lat/40.64/lon/-73.78/dist/100']  # 40.64 latitude, -73.78 longitude, 100 nautical miles
run_time_hours = 48
query_wait_time = 2  # adsb.fi public API limit is 1 request per second
temp_file = '../data/ads-b_extra.tmp.jsonl'
last_written = {}

# Restore deduplication state so a restarted run can continue the same temp file.
if os.path.isfile(temp_file):
    with open(temp_file, 'r+', encoding='utf-8') as temp_in:
        while True:
            line_start = temp_in.tell()
            line = temp_in.readline()
            if not line:
                break
            if not line.endswith('\n'):
                temp_in.seek(line_start)
                temp_in.truncate()
                break
            try:
                previous = json.loads(line)
                key = (previous['flight'], previous['lat'], previous['lon'])
                last_written[key] = float(previous['ts'])
            except (json.JSONDecodeError, KeyError, TypeError, ValueError):
                continue

total_time = 0
while total_time < (run_time_hours * 60 * 60):
    total_time = time.time() - start_time
    try:
        response = subprocess.run(command, capture_output=True, check=True)
        out = json.loads(response.stdout)
        ctime = out.get('ctime') or out.get('now')  # Preserve the API epoch-millisecond timestamp in ts
        if ctime is None:
            raise KeyError('API response is missing both ctime and now')
        with open(temp_file, 'a', encoding='utf-8') as temp_out:
            for item in out.get('ac', []):
                try:
                    part1 = (item.get('flight') or '') + (item.get('r') or '')
                    if not part1:
                        continue
                    key = (part1, item['lat'], item['lon'])
                    previous_ts = last_written.get(key)
                    if previous_ts is None or ctime - previous_ts > 30 * 60 * 1000:
                        item['flight'] = part1
                        item['ts'] = ctime
                        temp_out.write(json.dumps(item, separators=(',', ':')) + '\n')
                        temp_out.flush()
                        last_written[key] = ctime
                except (KeyError, TypeError, ValueError):
                    continue
    except Exception as error:
        print(f'Fetch or write failed: {error}')
    time.sleep(query_wait_time)

In [ ]:
temp_file = '../data/ads-b_extra.tmp.jsonl'
output_file = '../data/ads-b_extra.csv'
temp_db = '../data/ads-b_extra_temp.db'
aircraft_batch_size = 250  # Number of aircraft to batch together; keeps peak RAM < 100 MB

canonical_columns = [
    'hex', 'type', 'flight', 'r', 't', 'dbFlags', 'desc', 'ownOp', 'year',
    'alt_baro', 'alt_geom', 'gs', 'track', 'baro_rate', 'squawk', 'emergency',
    'category', 'nav_qnh', 'nav_altitude_mcp', 'lat', 'lon', 'nic', 'rc',
    'seen_pos', 'version', 'nic_baro', 'nac_p', 'nac_v', 'sil', 'sil_type',
    'gva', 'sda', 'alert', 'spi', 'mlat', 'tisb', 'messages', 'seen', 'rssi',
    'dst', 'dir', 'ts', 'ias', 'tas', 'mach', 'wd', 'ws', 'oat', 'tat',
    'track_rate', 'roll', 'mag_heading', 'true_heading', 'geom_rate',
    'nav_heading', 'nav_altitude_fms', 'nav_modes', 'calc_track',
    'gpsOkBefore', 'gpsOkLat', 'gpsOkLon', 'acas_ra'
]

if os.path.isfile(temp_file):
    if os.path.isfile(temp_db):
        os.remove(temp_db)

    write_header = not os.path.isfile(output_file) or os.path.getsize(output_file) == 0
    if write_header:
        output_columns = canonical_columns
    else:
        # Reuse the existing header order; API object key order is not stable.
        output_columns = pd.read_csv(output_file, nrows=0).columns.tolist()

    print('Indexing raw lines into temporary store...')
    conn = sqlite3.connect(temp_db)
    try:
        conn.execute('PRAGMA synchronous = OFF')
        conn.execute('PRAGMA journal_mode = OFF')
        conn.execute('CREATE TABLE records (hex TEXT, data TEXT)')

        batch = []
        with open(temp_file, 'r', encoding='utf-8') as temp_in:
            for line in temp_in:
                idx = line.find('"hex":"')
                h = line[idx+7:line.find('"', idx+7)] if idx != -1 else ''
                batch.append((h, line))
                if len(batch) >= 50000:
                    conn.executemany('INSERT INTO records VALUES (?, ?)', batch)
                    batch.clear()
            if batch:
                conn.executemany('INSERT INTO records VALUES (?, ?)', batch)
        conn.commit()
        conn.execute('CREATE INDEX idx_hex ON records(hex)')

        cursor = conn.cursor()
        distinct_hexes = [r[0] for r in cursor.execute('SELECT DISTINCT hex FROM records WHERE hex != ""').fetchall()]
        print(f'Found {len(distinct_hexes)} distinct aircraft. Writing to CSV in batches of {aircraft_batch_size} aircraft...')

        total_written = 0
        with open(output_file, 'a', newline='', encoding='utf-8') as f_out:
            for i in range(0, len(distinct_hexes), aircraft_batch_size):
                ac_batch = distinct_hexes[i:i + aircraft_batch_size]
                placeholders = ','.join('?' * len(ac_batch))
                cursor.execute(f'SELECT data FROM records WHERE hex IN ({placeholders}) ORDER BY hex', ac_batch)
                batch_rows = []
                for r in cursor.fetchall():
                    try:
                        batch_rows.append(json.loads(r[0]))
                    except json.JSONDecodeError:
                        continue
                if not batch_rows:
                    continue

                df_batch = pd.DataFrame(batch_rows)
                unexpected_columns = [col for col in df_batch.columns if col not in output_columns]
                if unexpected_columns:
                    raise ValueError(f'New API fields need schema review before append: {unexpected_columns}')
                df_batch = df_batch.reindex(columns=output_columns)
                df_batch.to_csv(f_out, index=False, header=write_header)
                write_header = False
                total_written += len(df_batch)

                processed_count = min(i + aircraft_batch_size, len(distinct_hexes))
                if (i // aircraft_batch_size) % 10 == 0 or processed_count == len(distinct_hexes):
                    print(f'  Processed {processed_count}/{len(distinct_hexes)} aircraft ({total_written:,} records written)...', flush=True)

            # Handle any records with missing or empty hex code
            cursor.execute('SELECT data FROM records WHERE hex IS NULL OR hex = ""')
            missing_rows = []
            for r in cursor.fetchall():
                try:
                    missing_rows.append(json.loads(r[0]))
                except json.JSONDecodeError:
                    continue
            if missing_rows:
                df_missing = pd.DataFrame(missing_rows)
                unexpected_columns = [col for col in df_missing.columns if col not in output_columns]
                if unexpected_columns:
                    raise ValueError(f'New API fields need schema review before append: {unexpected_columns}')
                df_missing = df_missing.reindex(columns=output_columns)
                df_missing.to_csv(f_out, index=False, header=write_header)
                write_header = False
                total_written += len(df_missing)

        print(f'Done! Successfully written {total_written:,} records to {output_file}.')
        os.remove(temp_file)
    finally:
        conn.close()
        if os.path.isfile(temp_db):
            os.remove(temp_db)

    df = pd.DataFrame(columns=output_columns)


In [ ]:
df.columns

In [ ]:
# out_file.close()# Old